<a href="https://colab.research.google.com/github/mahb97/punctuation-joyce-only-/blob/feature%2Fjoyce-punct-heatmaps/Punctuation_Geometries_in_Joyce_A_Calhoun_Inspired_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Joyce Punctuation: Calhoun-Style Visuals + Unicode Heatmaps

This notebook does two things:

1. Reproduces Adam J. Calhoun’s “punctuation raster” aesthetic by running his original script (glyph grid) on *Dubliners*, *A Portrait of the Artist as a Young Man*, *Ulysses*, and *Finnegans Wake*.  
2. Generates Unicode-aware punctuation density heatmaps using sliding windows over the texts:
   - Whole-book, single-row panels (classic look).
   - Per-chapter/per-story/per-episode grids (rows = parts, columns = progress).

**Inspiration/credit:** Adam J. Calhoun, “Punctuation in Novels” (2016).  
Medium: https://medium.com/@neuroecology/punctuation-in-novels-8f316d542ec4  
Original code repository: https://github.com/adamjcalhoun/punctuation


## Notes

The heatmap code here is a fresh implementation (Unicode-aware; no fonts), inspired by the original medium post. The original Calhoun script is used here for homage/glyph plots; see his repo for license and details. Texts are not tracked in git; place `.txt` files in a local `texts/` folder or Colab workspace.

## Inputs
- Local plain-text files:  
  `texts/dubliners.txt`, `texts/portrait.txt`, `texts/ulysses.txt`, `texts/finnegans_wake.txt`

## Outputs
- `out_heat/` PNG panels (whole-book + per-part grids) and matching CSVs of normalized densities.
- Optional `out/` glyph rasters from the original script.

## How to run (Colab)
1. Upload the four `.txt` files to `texts/`.
2. Run the cells in order:
   - Setup + (optional) original glyph runner.
   - Heatmap generator(s): whole-book panel; by-story (Dubliners); by-chapter (Portrait); by-episode (Ulysses); Wake whole-book.
3. Find images under `out_heat/`.



##**Attribution/Credits**

Glyph raster idea and original code by Adam J. Calhoun (“Punctuation in Novels,” 2016).  

This notebook’s heatmap implementation is independent but inspired by Calhoun’s visualization from the 2016 article: https://medium.com/@neuroecology/punctuation-in-novels-8f316d542ec4

Adam J. Calhoun, “Punctuation in Novels,” Medium, February 15, 2016. https://medium.com/@neuroecology/punctuation-in-novels-8f316d542ec4

Adam J. Calhoun, punctuation (source code), GitHub repository. https://github.com/adamjcalhoun/punctuation.









In [ ]:
import os
BASE = "/content/joyce_punct"
OUT  = f"{BASE}/out"
os.makedirs(BASE, exist_ok=True)
os.makedirs(OUT, exist_ok=True)
print("BASE:", BASE)
print("OUT :", OUT)

In [15]:
from google.colab import files, output
print("needs punctuation.py")
up = files.upload()
for name, data in up.items():
    with open(f"{BASE}/{name}", "wb") as f: f.write(data)
    print("Saved:", f"{BASE}/{name}")

needs punctuation.py


Saving punctuation.py to punctuation (2).py
Saved: /content/joyce_punct/punctuation (2).py


In [27]:
from google.colab import files, output
print("Upload GlacialIndifference-Bold.otf (or any .ttf/.otf you want to use)")
files.upload()

Upload GlacialIndifference-Bold.otf (or any .ttf/.otf you want to use)


Saving GlacialIndifference-Bold.otf to GlacialIndifference-Bold.otf


{'GlacialIndifference-Bold.otf': b'OTTO\x00\r\x00\x80\x00\x03\x00PCFF I\x97\xd60\x00\x00:\xe0\x00\x009\xceFFTMl\xc9\xf3\xb5\x00\x00x\x08\x00\x00\x00\x1cGDEF\x00\xeb\x00$\x00\x00t\xb0\x00\x00\x00(GPOS+\xb25\x14\x00\x00u\x08\x00\x00\x02\xfeGSUB\xb8\xff\xb8\xfe\x00\x00t\xd8\x00\x00\x000OS/2i\nbj\x00\x00\x01@\x00\x00\x00`cmap\xe6p!-\x00\x008\xb4\x00\x00\x02\nhead\x07\xbbr\xc0\x00\x00\x00\xdc\x00\x00\x006hhea\x06\xf1\x03\x7f\x00\x00\x01\x14\x00\x00\x00$hmtx\x8e\xf5\x16\x1d\x00\x00x$\x00\x00\x02\xf0maxp\x00\xbcP\x00\x00\x00\x018\x00\x00\x00\x06name\xaa9\\\x18\x00\x00\x01\xa0\x00\x007\x14post\xffr\x002\x00\x00:\xc0\x00\x00\x00 \x00\x01\x00\x00\x00\x01\x00Ac\xa1K5_\x0f<\xf5\x00\x0b\x03\xe8\x00\x00\x00\x00\xd2\xb1\x97o\x00\x00\x00\x00\xd2\xb1\x97o\xff\xac\xfe\xfd\x03\x8c\x03\xbf\x00\x01\x00\x08\x00\x02\x00\x00\x00\x00\x00\x00\x00\x01\x00\x00\x03\xb6\xff\x06\x00\x00\x04\x10\xff\xac\xff\xac\x03\x8c\x00\x01\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xbc\x00\x00P\x00\x00\xbc\x00\x0

In [30]:
from PIL import ImageFont
FONT = "/content/joyce_punct/GlacialIndifference-Bold.otf"
_ = ImageFont.truetype(FONT, 20)
print("Loaded OK.")

Loaded OK.


In [32]:
# Normalise to ASCII
import re, pathlib, shutil

BASE = "/content/joyce_punct"; CLEAN = f"{BASE}/clean_ascii"
os.makedirs(CLEAN, exist_ok=True)

def ascii_punct(s: str) -> str:
    # map common Unicode punct to ASCII
    s = (s.replace("“", '"').replace("”", '"')
           .replace("‘", "'").replace("’", "'")
           .replace("—", "-").replace("–", "-")
           .replace("…", "..."))
    return s

inputs = {
    "Dubliners":      f"{BASE}/dubliners.txt",
    "Portrait":       f"{BASE}/portrait.txt",
    "Ulysses":        f"{BASE}/ulysses.txt",
    "FinnegansWake":  f"{BASE}/finnegans_wake.txt",
}

norm_paths = {}
for label, path in inputs.items():
    txt = pathlib.Path(path).read_text(encoding="utf-8", errors="ignore")
    txt = ascii_punct(txt)
    outp = f"{CLEAN}/{label}.txt"
    pathlib.Path(outp).write_text(txt, encoding="utf-8")
    norm_paths[label] = outp
    print("Normalised ->", outp)

Normalised -> /content/joyce_punct/clean_ascii/Dubliners.txt
Normalised -> /content/joyce_punct/clean_ascii/Portrait.txt
Normalised -> /content/joyce_punct/clean_ascii/Ulysses.txt
Normalised -> /content/joyce_punct/clean_ascii/FinnegansWake.txt


In [33]:
import os, sys, glob, shutil
from pathlib import Path
from PIL import ImageFont

BASE = "/content/joyce_punct"
OUT  = f"{BASE}/out"
os.makedirs(OUT, exist_ok=True)

PUNCT_PATH = Path(BASE) / "punctuation.py"
assert PUNCT_PATH.exists(), "punctuation.py missing."

ImageFont.truetype(str(Path(BASE)/"GlacialIndifference-Bold.otf"), 10)

def run_original_adhoc(text_path: str, label: str,
                       symbols=80, lines=80,
                       canvas_w=1200, canvas_h=1200, trim=0,
                       out_png=None):
    text_path = Path(text_path); assert text_path.exists()
    dst_txt = Path(BASE)/f"{label}.txt"
    shutil.copyfile(text_path, dst_txt)

    g = {"__name__": "__main__", "sys": sys}

    g["bookname"]        = label
    g["symbolsPerLine"]  = int(symbols)
    g["linesOfText"]     = int(lines)
    g["canvasWidth"]     = int(canvas_w)
    g["canvasHeight"]    = int(canvas_h)
    g["trim"]            = int(trim)

    g["font1size"] = 36          # title
    g["font2size"] = 20          # body glyphs
    g["endSentenceFill"]   = (0,0,0)  # periods/?! in black
    g["transitionFill"]    = (0,0,0)  # ,;:
    g["parentheticalFill"] = (0,0,0)  # quotes/brackets/dashes

    # if script does open(bookname + ".txt"), pass only the stem:
    sys.argv = [str(PUNCT_PATH), label]

    code = PUNCT_PATH.read_text(encoding="utf-8", errors="ignore")

    cwd = os.getcwd()
    try:
        before = set(glob.glob(str(Path(BASE)/"*.png")))
        os.chdir(BASE)
        exec(compile(code, str(PUNCT_PATH), "exec"), g, g)
        after = set(glob.glob(str(Path(BASE)/"*.png")))
    finally:
        os.chdir(cwd)

    created = sorted((after - before), key=lambda p: os.path.getmtime(p))
    if not created:
        print(f"[{label}] no PNG produced."); return None
    produced = created[-1]
    out_png = Path(out_png) if out_png else (Path(OUT)/f"{label}.png")
    out_png.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(produced, out_png)
    print(f"[{label}] -> {out_png}")
    return str(out_png)

texts = {
    "Dubliners":      f"{BASE}/dubliners.txt",
    "Portrait":       f"{BASE}/portrait.txt",
    "Ulysses":        f"{BASE}/ulysses.txt",
    "FinnegansWake":  f"{BASE}/finnegans_wake.txt",
}

for label, path in norm_paths.items():
    run_original_adhoc(
        text_path=path,
        label=label,
        symbols=80, lines=80,
        canvas_w=1200, canvas_h=1200,
        trim=0,
        out_png=f"{OUT}/{label}.png"
    )

print("Done. Check:", OUT)


14558
[Dubliners] no PNG produced.
13332
[Portrait] no PNG produced.
60872
[Ulysses] no PNG produced.
49911
[FinnegansWake] no PNG produced.
Done. Check: /content/joyce_punct/out


In [34]:
import os, re, unicodedata, csv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

BASE = "/content/joyce_punct"
TEXTS = {
    "Dubliners":       f"{BASE}/dubliners.txt",
    "Portrait":        f"{BASE}/portrait.txt",
    "Ulysses":         f"{BASE}/ulysses.txt",
    "Finnegans Wake":  f"{BASE}/finnegans_wake.txt",
}
OUT = f"{BASE}/out_heat"
os.makedirs(OUT, exist_ok=True)

WINDOW = 256
STRIDE = 64
CMAP = "gray"
DPI = 220

def is_punct(ch):
    return unicodedata.category(ch).startswith("P")

def sliding_windows(n, win, stride):
    if n <= win: return [(0, n)]
    starts = list(range(0, n - win + 1, stride))
    if starts[-1] + win < n: starts.append(n - win)
    return [(s, s + win) for s in starts]

def build_matrix(text):
    chars = list(text)
    n = len(chars)
    wins = sliding_windows(n, WINDOW, STRIDE)

    classes = [
        ("Any", lambda c: is_punct(c)),
        (".?!", lambda c: c in ".?!"),
        (",;:", lambda c: c in ",;:"),
        ("Quotes", lambda c: c in "\"'“”‘’"),
        ("Dashes", lambda c: c in "-–—"),
        ("Parens", lambda c: c in "()[]{}"),
    ]

    rows = []
    names = []
    for name, pred in classes:
        mask = np.fromiter((pred(c) for c in chars), dtype=np.uint8, count=n)
        row = []
        for a, b in wins:
            seg = mask[a:b]
            row.append(seg.mean() if seg.size else 0.0)
        rows.append(row)
        names.append(name)

    M = np.array(rows, dtype=np.float32)  # [R, C]
    for r in range(M.shape[0]):
        mn, mx = M[r].min(), M[r].max()
        M[r] = 0.0 if mx - mn < 1e-12 else (M[r] - mn) / (mx - mn)
    return M, names

def save_heatmap(M, names, title, out_png):
    plt.figure(figsize=(min(14, 0.18*M.shape[1] + 3), 2 + 0.5*M.shape[0]))
    plt.imshow(M, aspect="auto", cmap=CMAP, interpolation="nearest")
    plt.yticks(range(len(names)), names, fontsize=9)
    plt.xticks([], [])
    plt.title(title, fontname="Times New Roman", fontsize=12)
    plt.tight_layout()
    plt.savefig(out_png, dpi=DPI)
    plt.close()

panels = []
for label, path in TEXTS.items():
    txt = Path(path).read_text(encoding="utf-8", errors="ignore")
    txt = re.sub(r'\r\n?', '\n', txt)
    txt = re.sub(r'(?s)\*\*\*.*?\*\*\*', '', txt)
    M, names = build_matrix(txt)
    png = f"{OUT}/{label.replace(' ','_')}_punct_heat.png"
    save_heatmap(M, names, f"{label} — Punctuation Density (win={WINDOW}, stride={STRIDE})", png)
    panels.append(png)
    print("Saved:", png)

# 2×2 panel
imgs = [Image.open(p).convert("RGB") for p in panels]
h = min(i.height for i in imgs); imgs = [i.resize((int(i.width*h/i.height), h)) for i in imgs]
w = max(i.width for i in imgs)
def pad(i, w):
    from PIL import Image as _I;
    if i.width==w: return i
    c=_I.new("RGB",(w,i.height),(255,255,255)); c.paste(i,(0,0)); return c
imgs = [pad(i,w) for i in imgs]
from PIL import Image as I
panel = I.new("RGB", (w*2, h*2), (255,255,255))
for i,(x,y) in enumerate([(0,0),(w,0),(0,h),(w,h)]):
    panel.paste(imgs[i], (x,y))
panel_path = f"{OUT}/joyce_punctuation_panel.png"
panel.save(panel_path); print("Saved panel:", panel_path)

Saved: /content/joyce_punct/out_heat/Dubliners_punct_heat.png


Saved: /content/joyce_punct/out_heat/Portrait_punct_heat.png


Saved: /content/joyce_punct/out_heat/Ulysses_punct_heat.png


Saved: /content/joyce_punct/out_heat/Finnegans_Wake_punct_heat.png
Saved panel: /content/joyce_punct/out_heat/joyce_punctuation_panel.png


In [45]:
import re, itertools
from pathlib import Path

BASE = "/content/joyce_punct"
portrait_path = Path(f"{BASE}/portrait.txt")
txt = portrait_path.read_text(encoding="utf-8", errors="ignore")

# normalise newlines + strip Gutenberg boilerplate if present
import re as _re
txt = _re.sub(r'\r\n?', '\n', txt)
txt = _re.sub(r'(?s)\*\*\*.*?\*\*\*', '', txt)

lines = txt.splitlines()

# Try a few header shapes
candidates = []
patterns = [
    r'^\s*CHAPTER\s+[IVX]+\s*\.?\s*$',
    r'^\s*Chapter\s+[IVX]+\s*\.?\s*$',
    r'^\s*CHAPTER\s+\d+\s*\.?\s*$',
    r'^\s*Chapter\s+\d+\s*\.?\s*$',
    r'^\s*(I|II|III|IV|V)\s*$',
]
for i, line in enumerate(lines, start=1):
    for pat in patterns:
        if re.match(pat, line):
            candidates.append((i, line.strip()))
            break

print(f"Found {len(candidates)} possible headings:")
for ln, s in candidates[:20]:
    print(f"{ln:>6}: {s}")

Found 5 possible headings:
    13: Chapter I
  2296: Chapter II
  3893: Chapter III
  5519: Chapter IV
  6468: Chapter V


In [46]:
import numpy as np

def split_portrait_by_lines(text: str, header_lines: list[int]):
    lines = text.splitlines()
    idxs = sorted(set(int(i) for i in header_lines if 1 <= i <= len(lines)))
    if not idxs:
        return []

    chunks = []
    for i, start_ln in enumerate(idxs):
        start = start_ln - 1
        end = (idxs[i+1]-1) if i+1 < len(idxs) else len(lines)
        chunks.append("\n".join(lines[start:end]).strip())
    return chunks

header_lines = [ln for ln,_ in candidates]

if len(header_lines) >= 5:
    header_lines = header_lines[:5]

portrait_chunks = split_portrait_by_lines(txt, header_lines)
print("Chapters extracted:", len(portrait_chunks))
for i, ch in enumerate(portrait_chunks, 1):
    print(f"Ch {i}: {len(ch):,} chars")

Chapters extracted: 5
Ch 1: 93,463 chars
Ch 2: 79,961 chars
Ch 3: 90,354 chars
Ch 4: 52,786 chars
Ch 5: 164,276 chars


In [47]:
grid_for_chunks(
    portrait_chunks,
    "Portrait — Punctuation Density by Chapter",
    f"{OUT}/Portrait_by_chapter.png"
)

Saved grid: /content/joyce_punct/out_heat/Portrait_by_chapter.png


In [50]:
# Dubliners: punctuation density by STORY
import os, re, unicodedata, csv
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE = "/content/joyce_punct"
TXT  = f"{BASE}/dubliners.txt"
OUT  = f"{BASE}/out_heat"; os.makedirs(OUT, exist_ok=True)

# dubs stories in canonical order
TITLES = [
    "The Sisters","An Encounter","Araby","Eveline","After the Race","Two Gallants",
    "The Boarding House","A Little Cloud","Counterparts","Clay","A Painful Case",
    "Ivy Day in the Committee Room","A Mother","Grace","The Dead"
]

BINS = 400           # columns per story
CMAP = "binary"      # black on white
DPI  = 220

def is_punct(ch): return unicodedata.category(ch).startswith("P")

def punct_density_binned(s: str, bins: int) -> np.ndarray:
    n = len(s)
    if n == 0: return np.zeros(bins, dtype=np.float32)
    mask = np.fromiter((is_punct(c) for c in s), dtype=np.uint8, count=n)
    edges = np.linspace(0, n, bins + 1, dtype=int)
    out = np.zeros(bins, dtype=np.float32)
    for b in range(bins):
        a, c = edges[b], edges[b+1]
        seg = mask[a:c]
        out[b] = float(seg.mean()) if seg.size else 0.0
    return out

# load + tidy
txt = Path(TXT).read_text(encoding="utf-8", errors="ignore")
txt = re.sub(r'\r\n?', '\n', txt)
txt = re.sub(r'(?s)\*\*\*.*?\*\*\*', '', txt)

# find titles in-order (skips TOC)
def find_titles_in_order(text, titles):
    pos, cursor = [], 0
    for t in titles:
        m = re.search(rf'(?mi)^\s*{re.escape(t)}\s*$', text[cursor:])
        if not m:
            m = re.search(rf'(?mi)^\s*{re.escape(t)}\s*[\.\-—–]*\s*$', text[cursor:])
        if m:
            idx = cursor + m.start()
            pos.append((t, idx))
            cursor = idx + 1
        else:
            pos.append((t, None))
    return pos

positions = find_titles_in_order(txt, TITLES)
found = [(t,i) for t,i in positions if i is not None]
names = [t for t,_ in found]
idxs  = [i for _,i in found]

# slice stories
segments = []
for k, start in enumerate(idxs):
    end = idxs[k+1] if k+1 < len(idxs) else len(txt)
    segments.append(txt[start:end])

# build matrix: one row per story
rows = [punct_density_binned(seg, BINS) for seg in segments]
M = np.vstack(rows)

# per-row min–max for contrast
X = M.copy()
for r in range(X.shape[0]):
    mn, mx = X[r].min(), X[r].max()
    X[r] = 0.0 if mx - mn < 1e-12 else (X[r] - mn) / (mx - mn)

# plot
plt.figure(figsize=(18, 6))
ax = plt.gca()
ax.imshow(X, aspect="auto", cmap=CMAP, interpolation="nearest")
ax.set_yticks(range(len(names)))
ax.set_yticklabels([str(i+1) for i in range(len(names))], fontsize=9)
ax.set_xticks([])
ax.set_title("Dubliners — Punctuation Density by Story", fontsize=14)
for y in range(1, len(names)): ax.axhline(y-0.5, linewidth=0.3, color="0.85")
ax2 = ax.twinx(); ax2.set_ylim(ax.get_ylim())
ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names, fontsize=8)
ax2.tick_params(axis="y", which="major", pad=6)

plt.tight_layout()
png_path = f"{OUT}/Dubliners_by_story_gray.png"
plt.savefig(png_path, dpi=DPI, bbox_inches="tight"); plt.close()
print("Saved:", png_path)

# optional CSV of the normalized matrix
with open(f"{OUT}/Dubliners_by_story_gray.csv","w",newline="",encoding="utf-8") as f:
    import csv; w=csv.writer(f)
    w.writerow(["story"]+[f"col_{i}" for i in range(BINS)])
    for title,row in zip(names,X): w.writerow([title]+[f"{v:.6f}" for v in row])

Saved: /content/joyce_punct/out_heat/Dubliners_by_story_gray.png


In [54]:
# Ulysses
import os, re, unicodedata, csv
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE = "/content/joyce_punct"
TXT  = f"{BASE}/ulysses.txt"
OUT  = f"{BASE}/out_heat"; os.makedirs(OUT, exist_ok=True)

EPISODES = list(range(1, 19))
BINS = 400
CMAP = "binary"
DPI  = 220

def is_punct(ch):  # Unicode-aware
    return unicodedata.category(ch).startswith("P")

def punct_density_binned(s: str, bins: int) -> np.ndarray:
    n = len(s)
    if n == 0: return np.zeros(bins, dtype=np.float32)
    mask = np.fromiter((is_punct(c) for c in s), dtype=np.uint8, count=n)
    edges = np.linspace(0, n, bins + 1, dtype=int)
    out = np.zeros(bins, dtype=np.float32)
    for b in range(bins):
        a, c = edges[b], edges[b+1]
        seg = mask[a:c]
        out[b] = float(seg.mean()) if seg.size else 0.0
    return out

# load + tidy
txt = Path(TXT).read_text(encoding="utf-8", errors="ignore")
txt = re.sub(r'\r\n?', '\n', txt)
txt = re.sub(r'(?s)\*\*\*.*?\*\*\*', '', txt)

# find markers
def find_episode_positions(text: str, episodes: list[int]):
    pos = []
    cursor = 0
    for k in episodes:
        # primary: [ 12 ] / [12] at line start
        pat = rf'(?m)^\s*\[\s*{k}\s*\]\s*$'
        m = re.search(pat, text[cursor:])
        if not m:
            # fallback
            pat2 = rf'(?mi)^\s*Episode\s+({k}|X?V?I{0,3}|IX|IV|V?I{0,3})\s*$'
            m = re.search(pat2, text[cursor:])
        if m:
            idx = cursor + m.start()
            pos.append((k, idx))
            cursor = idx + 1
        else:
            pos.append((k, None))
    return pos

positions = find_episode_positions(txt, EPISODES)
found = [(k,i) for k,i in positions if i is not None]
missing = [k for k,i in positions if i is None]
print(f"Detected {len(found)}/18 episode markers.")
if missing: print("Missing:", missing)

# slice episodes using found indices
idxs = [i for _,i in found]
nums = [k for k,_ in found]
segments = []
for j, start in enumerate(idxs):
    end = idxs[j+1] if j+1 < len(idxs) else len(txt)
    segments.append(txt[start:end])

# compute matrix (one row per detected episode)
rows = [punct_density_binned(seg, BINS) for seg in segments]
M = np.vstack(rows) if rows else np.zeros((0, BINS), dtype=np.float32)

# per-row min–max for contrast
X = M.copy()
for r in range(X.shape[0]):
    mn, mx = X[r].min(), X[r].max()
    X[r] = 0.0 if mx - mn < 1e-12 else (X[r] - mn) / (mx - mn)

# plot
plt.figure(figsize=(18, 7))
ax = plt.gca()
ax.imshow(X, aspect="auto", cmap=CMAP, interpolation="nearest")
ax.set_yticks(range(len(nums)))
ax.set_yticklabels([str(n) for n in nums], fontsize=9)
ax.set_xticks([])
ax.set_title("Ulysses — Punctuation Density by Episode", fontsize=14)
for y in range(1, len(nums)):
    ax.axhline(y - 0.5, linewidth=0.3, color="0.85")

plt.tight_layout()
png_path = f"{OUT}/Ulysses_by_episode_gray.png"
plt.savefig(png_path, dpi=DPI, bbox_inches="tight")
plt.close()
print("Saved:", png_path)
# save
csv_path = f"{OUT}/Ulysses_by_episode_gray.csv"
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["episode"] + [f"col_{i}" for i in range(BINS)])
    for n, row in zip(nums, X):
        w.writerow([n] + [f"{v:.6f}" for v in row])
print("Saved:", csv_path)

Detected 18/18 episode markers.
Saved: /content/joyce_punct/out_heat/Ulysses_by_episode_gray.png
Saved: /content/joyce_punct/out_heat/Ulysses_by_episode_gray.csv


In [55]:
# fw punctuation density
import os, re, unicodedata, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

BASE = "/content/joyce_punct"
TXT  = f"{BASE}/finnegans_wake.txt"
OUT  = f"{BASE}/out_heat"; os.makedirs(OUT, exist_ok=True)

BINS = 1200
CMAP = "binary"
DPI  = 220

def is_punct(ch): return unicodedata.category(ch).startswith("P")

def punct_density_binned(s: str, bins: int) -> np.ndarray:
    n = len(s)
    if n == 0: return np.zeros(bins, dtype=np.float32)
    mask = np.fromiter((is_punct(c) for c in s), dtype=np.uint8, count=n)
    edges = np.linspace(0, n, bins + 1, dtype=int)
    out = np.zeros(bins, dtype=np.float32)
    for b in range(bins):
        a, c = edges[b], edges[b+1]
        seg = mask[a:c]
        out[b] = float(seg.mean()) if seg.size else 0.0
    return out

txt = Path(TXT).read_text(encoding="utf-8", errors="ignore")
txt = re.sub(r'\r\n?', '\n', txt)
txt = re.sub(r'(?s)\*\*\*.*?\*\*\*', '', txt)

row = punct_density_binned(txt, BINS)[None, :]   # shape [1, C]
# row-wise min–max (keeps contrast nice)
mn, mx = row.min(), row.max()
X = np.zeros_like(row) if mx-mn < 1e-12 else (row - mn)/(mx-mn)

plt.figure(figsize=(18, 2.5))
plt.imshow(X, aspect="auto", cmap=CMAP, interpolation="nearest")
plt.yticks([0], ["Any"])
plt.xticks([])
plt.title("Finnegans Wake — Punctuation Density", fontsize=14)
plt.tight_layout()
png_path = f"{OUT}/Finnegans_Wake_heat_whole.png"
plt.savefig(png_path, dpi=DPI, bbox_inches="tight"); plt.close()
print("Saved:", png_path)

Saved: /content/joyce_punct/out_heat/Finnegans_Wake_heat_whole.png
